In [13]:
from datasets import load_dataset
import json
from tqdm import tqdm
import re

In [14]:
class GSM8KConfig:
    # Model
    model_name = "Qwen/Qwen3-8B"
    
    # P-Tuning
    num_virtual_tokens = 20
    
    # System Prompt (要被压缩的部分)
    system_prompt = """You are a mathematical reasoning assistant. Follow these steps:
1. Read the problem carefully
2. Break it down into steps
3. Show your work clearly
4. Provide the final numerical answer

Format your response as:
Reasoning: [your step-by-step solution]
Answer: [numerical answer only]"""
    
    # Training
    batch_size = 8
    gradient_accumulation_steps = 2
    learning_rate = 1e-2
    num_epochs = 5
    max_length = 1024  # GSM8K 需要较长context
    temperature = 2.0
    
    # Data
    train_size = 7473  # Full GSM8K train set
    eval_size = 500    # Subset of test set
    
    # Output
    output_dir = "output/qwen3_ptuning_gsm8k"
    save_steps = 200
    eval_steps = 200
    logging_steps = 10
    
    # Hardware
    fp16 = True
    seed = 42

In [27]:
class GSM8KDataset:
    def __init__(self, config: GSM8KConfig, tokenizer, split="train"):
        self.config = config
        self.tokenizer = tokenizer
        
        # Load GSM8K dataset
        dataset = load_dataset("gsm8k", "main")
        
        if split == "train":
            self.data = dataset["train"]
            if config.train_size < len(self.data):
                self.data = self.data.select(range(config.train_size))
        else:
            self.data = dataset["test"]
            if config.eval_size < len(self.data):
                self.data = self.data.select(range(config.eval_size))
        
        print(f"✅ Loaded {len(self.data)} {split} samples")
    
    def __len__(self):
        return len(self.data)
    
    def extract_answer(self, answer_text):
        """Extract numerical answer from GSM8K format"""
        # GSM8K format: "#### 42"
        match = re.search(r'#### (.+)', answer_text)
        if match:
            return match.group(1).strip()
        return answer_text.strip()
    
    def __getitem__(self, idx):
        item = self.data[idx]
        question = item["question"]
        full_answer = item["answer"]
        
        # Extract numerical answer
        numerical_answer = self.extract_answer(full_answer)
        
        # Extract reasoning steps (everything before ####)
        reasoning = result = re.sub(r'<<.*?>>', '', full_answer.split("####")[0].strip())
        
        # Teacher input: system + question
        teacher_messages = [
            {"role": "system", "content": self.config.system_prompt},
            {"role": "user", "content": question},
            {"role": "assistant", "content": f"Reasoning: {reasoning}\nAnswer: {numerical_answer}"}
        ]
        
        return {
            "id": f"gsm8k-{idx}",
            "conversation": teacher_messages
        }

In [31]:
config = GSM8KConfig()

dataset = GSM8KDataset(config=config, tokenizer=None, split="test")

✅ Loaded 500 test samples


In [32]:
dataset[0]

{'id': 'gsm8k-0',
 'conversation': [{'role': 'system',
   'content': 'You are a mathematical reasoning assistant. Follow these steps:\n1. Read the problem carefully\n2. Break it down into steps\n3. Show your work clearly\n4. Provide the final numerical answer\n\nFormat your response as:\nReasoning: [your step-by-step solution]\nAnswer: [numerical answer only]'},
  {'role': 'user',
   'content': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?"},
  {'role': 'assistant',
   'content': 'Reasoning: Janet sells 16 - 3 - 4 = 9 duck eggs a day.\nShe makes 9 * 2 = $18 every day at the farmer’s market.\nAnswer: 18'}]}

In [33]:
with open("../dataset/gsm8k/test.jsonl", "w") as f:
    for i in tqdm(range(len(dataset)), desc="Saving GSM8K train data"):
        data = dataset[i]
        f.write(json.dumps(data) + "\n")

Saving GSM8K train data: 100%|██████████| 500/500 [00:00<00:00, 7282.44it/s]
